<div style="background:#A6192E;color:#fff;padding:22px 28px;border-radius:14px;font-family:'Barlow Condensed','Arial Narrow',Arial,sans-serif">
<div style="letter-spacing:.2em;text-transform:uppercase;font-size:13px;opacity:.9">Python para Finanzas y Análisis de Datos · Sesión 1</div>
<div style="font-size:34px;font-weight:300;text-transform:uppercase;margin-top:6px"><span style="background:#fff;color:#A6192E;border-radius:10px;padding:2px 12px;margin-right:14px;font-weight:500">1.5</span>Backtesting honesto</div>
</div>

**Objetivo.** backtestear una estrategia sin hacer trampa y entender por qué optimizar sobre los mismos datos engaña.

**Lo que vamos a hacer**

- Programar un cruce de medias móviles evitando el *look-ahead bias*.
- Optimizar parámetros en entrenamiento y evaluar en prueba.
- Ver qué pasa al elegir la «mejor» de 200 estrategias sobre puro ruido.

In [ ]:
import sys, pathlib, subprocess, warnings
warnings.filterwarnings("ignore")

# Localiza la carpeta del curso (en Google Colab clona el repositorio)
root = pathlib.Path.cwd().resolve()
while root != root.parent and not (root / "common").exists():
    root = root.parent
if not (root / "common").exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/Ulysess824/Python-for-Finance"], check=False)
    root = pathlib.Path("Python-for-Finance/python-finance-course").resolve()
sys.path.insert(0, str(root))
# En Colab, ejecuta antes:  !pip install -q yfinance arch

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from common import *
set_plot_style()

## 1. La estrategia

In [ ]:
px = download_prices("SPY", start="2005-01-01").iloc[:, 0]
train, test = px[:"2016-12-31"], px["2017-01-01":]
print(f"Entrenamiento: {len(train)} días · Prueba: {len(test)} días")

Regla: estar **invertido** si la media móvil rápida > lenta, si no, en efectivo. La señal de hoy se opera **mañana** (`shift(1)`); si no, usamos información del futuro.

> ### ✏️ Ejercicio 1 — backtest
> Devuelve los retornos diarios de la estrategia, descontando `cost` por cada cambio de posición.

In [ ]:
def backtest(px, fast, slow, cost=0.0005):
    ret = px.pct_change().fillna(0)
    # TODO
    return ret * 0

In [ ]:
def backtest(px, fast, slow, cost=0.0005):
    ret = px.pct_change().fillna(0)
    signal = (px.rolling(fast).mean() > px.rolling(slow).mean()).astype(int)
    pos = signal.shift(1).fillna(0)                      # opera al día siguiente
    return pos * ret - cost * pos.diff().abs().fillna(0)

## 2. Optimizar en entrenamiento, evaluar en prueba

In [ ]:
rows = [(f, s, sharpe_ratio(backtest(train, f, s))) for f in range(10, 60, 10) for s in range(80, 260, 30)]
grid = pd.DataFrame(rows, columns=["fast", "slow", "sharpe"]).pivot(index="fast", columns="slow", values="sharpe")
f_best, s_best = grid.stack().idxmax()

fig, ax = plt.subplots(figsize=(8, 3.8))
im = ax.imshow(grid.values, cmap="Reds", aspect="auto")
ax.set_xticks(range(len(grid.columns)), grid.columns); ax.set_yticks(range(len(grid.index)), grid.index)
ax.set_xlabel("Media lenta"); ax.set_ylabel("Media rápida"); ax.grid(False)
ax.set_title("Sharpe en entrenamiento"); fig.colorbar(im); plt.show()

res = pd.DataFrame({
    "Entrenamiento": [sharpe_ratio(backtest(train, f_best, s_best)), sharpe_ratio(train.pct_change().dropna())],
    "Prueba":        [sharpe_ratio(backtest(test, f_best, s_best)),  sharpe_ratio(test.pct_change().dropna())],
}, index=[f"Cruce {f_best}/{s_best}", "Buy & hold"])
print("Sharpe:"); display(res.round(2))

## 3. El autoengaño estadístico

Generamos 2.500 días de retornos **sin ninguna señal** y 200 estrategias con posiciones aleatorias. Si elegimos la mejor en entrenamiento, ¿cómo le va en prueba?

> ### ✏️ Ejercicio 2 — Elegir al ganador sobre ruido
> Calcula el Sharpe anual de cada estrategia en entrenamiento (primer 70%) y en prueba (último 30%). Reporta el ganador de entrenamiento.

In [ ]:
rng = np.random.default_rng(0)
noise = rng.normal(0.0003, 0.01, 2500)                      # retornos sin ninguna señal
signals = rng.integers(0, 2, (200, len(noise)))             # 200 estrategias aleatorias
strat = signals * noise
cut = int(0.7 * len(noise))
# TODO: is_sh, os_sh = Sharpe anual por estrategia en entrenamiento y en prueba
is_sh = os_sh = np.zeros(200)

In [ ]:
rng = np.random.default_rng(0)
noise = rng.normal(0.0003, 0.01, 2500)
signals = rng.integers(0, 2, (200, len(noise)))
strat = signals * noise
cut = int(0.7 * len(noise))

def sharpe(a):
    return a.mean(axis=1) / a.std(axis=1) * np.sqrt(252)

is_sh, os_sh = sharpe(strat[:, :cut]), sharpe(strat[:, cut:])
b = is_sh.argmax()
print(f"Mejor en entrenamiento: Sharpe {is_sh[b]:.2f}  →  en prueba: {os_sh[b]:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(is_sh, os_sh, s=18, color="#5f5f5f", alpha=.7)
ax.scatter(is_sh[is_sh.argmax()], os_sh[is_sh.argmax()], s=90, color="#A6192E", label="«Ganadora»")
ax.axhline(0, color="#d9d9d9"); ax.axvline(0, color="#d9d9d9")
ax.set_xlabel("Sharpe en entrenamiento"); ax.set_ylabel("Sharpe en prueba")
ax.set_title("200 estrategias sin señal real"); ax.legend(); plt.show()

## 4. Costos de transacción

> ### ✏️ Ejercicio 3 — ¿Sobrevive a los costos?
> Compara el Sharpe de la mejor estrategia en `test` con costo 0 y con costo de 0,2% por operación.

In [ ]:
# TODO

In [ ]:
for c in (0, 0.002):
    print(f"costo {c:.1%}: Sharpe prueba = {sharpe_ratio(backtest(test, f_best, s_best, cost=c)):.2f}")

## Para llevarse

- Si la mejor estrategia sobre ruido luce bien en entrenamiento y mal en prueba, lo mismo puede pasarte con una real.
- Separa siempre entrenamiento y prueba, y evita el *look-ahead bias* con `shift`.
- Cuantas más combinaciones pruebas, más probable es encontrar un «ganador» falso.

---
*Excel para mirar, Python para repetir.*